In [2]:
pip install -U pymgrid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 7.8 MB/s eta 0:00:0000:0100:01m
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.4 MB/s eta 0:00:00
  Created wheel for pymgrid: filename=pymgrid-1.2.2-py3-none-any.whl size=3492849 sha256=843d408d970957e9be5c52e791cbe97ad5d8909dd7e3646a4e52f384f6fa950f
  Stored in directory: /root/.cache/pip/wheels/22/74/ee/35d2b7ad23381dc521b2c8670c6e30be0b2e9fe80a1fe03160
Successfully built pymgrid


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pymgrid import MicrogridGenerator as mg

plt.style.use('seaborn-v0_8-darkgrid')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



### Bins para carga neta

In [8]:
# 1. Cargamos los datos del simulador
env_pymgrid = mg(nb_microgrid=20, path="/data/pymgrid_data")
mg0 = env_pymgrid.microgrids[0]

# 2. Calculamos la carga neta de todo el año
carga_neta = mg0.load - mg0.pv

# 3. Calculamos los cuartiles (25%, 50%, 75%)
bins_carga = np.round(np.percentile(carga_neta, [25, 50, 75]), 2)

print("--- ANÁLISIS DE CARGA NETA ---")
print(f"Mínimo (Máx excedente solar): {carga_neta.min():.2f} kW")
print(f"Máximo (Pico consumo): {carga_neta.max():.2f} kW")
print(f"BINS RECOMENDADOS: {bins_carga.tolist()}")

# 4. Visualización
plt.figure(figsize=(10, 4))
plt.hist(carga_neta, bins=50, color='royalblue', edgecolor='black', alpha=0.7)
for b in bins_carga:
    plt.axvline(b, color='red', linestyle='dashed', linewidth=2, label=f'Bin {b}')
plt.title("Distribución Anual de la Carga Neta (Load - PV)")
plt.xlabel("kW (Negativo = Sobra energía, Positivo = Falta energía)")
plt.ylabel("Frecuencia (Horas)")
plt.legend()
plt.show()

IndexError: list index out of range

### Bins para precios

# 1. Cargamos el CSV (usamos sep='\t' por el formato que mostraste antes)
df_precios = pd.read_csv('datos_esios.csv', sep='\t')

# 2. Extraemos el valor y convertimos de €/MWh a €/kWh
precios_kwh = df_precios['value'].values / 1000.0

# 3. Calculamos los cuartiles (25%, 50%, 75%)
bins_precio = np.round(np.percentile(precios_kwh, [25, 50, 75]), 3)

print("--- ANÁLISIS DE PRECIOS DE ENERGÍA ---")
print(f"Precio Mínimo: {precios_kwh.min():.4f} €/kWh")
print(f"Precio Máximo: {precios_kwh.max():.4f} €/kWh")
print(f"BINS RECOMENDADOS: {bins_precio.tolist()}")

# 4. Visualización
plt.figure(figsize=(10, 4))
plt.hist(precios_kwh, bins=50, color='orange', edgecolor='black', alpha=0.7)
for b in bins_precio:
    plt.axvline(b, color='red', linestyle='dashed', linewidth=2, label=f'Bin {b}')
plt.title("Distribución Anual del Precio de la Energía (PVPC)")
plt.xlabel("Precio (€/kWh)")
plt.ylabel("Frecuencia (Horas)")
plt.legend()
plt.show()